### IMPORT AND SETUP

In [ ]:
import os
import sys
import torch
import gc
import pandas as pd
import numpy as np

# Khai báo đường dẫn root để import các module từ thư mục core
sys.path.append(os.path.abspath("../"))

from core.module02_fm_embedding_extraction.models import BioModelManager
from core.module02_fm_embedding_extraction.extractor import FeatureExtractor
from core.module03_geometry_extraction.extractor import LatentGeometryCalculator
from core.module04_bio_context_normalization.normalizer import AdvancedFeatureNormalizer

# Giải phóng bộ nhớ GPU trước khi chạy bộ khung trích xuất mới
torch.cuda.empty_cache()
gc.collect()

print(f"[+] CUDA khả dụng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] Đang sử dụng thiết bị: {torch.cuda.get_device_name(0)}")

### Matrix Configurations

In [ ]:
# Set up cho Module 2
# 1. Danh sách các file Parquet cần chạy (Hỗ trợ Train, Val, Test gom trong 1 lần cắm máy)
INPUT_FILES_CONFIG = [
    {
        "input_path": "D:/variant_data/train_split.parquet",
        "output_dir": "D:/variant_data/embeddings/train"
    },
    {
        "input_path": "D:/variant_data/val_split.parquet",
        "output_dir": "D:/variant_data/embeddings/val"
    },
    {
        "input_path": "D:/variant_data/test_clinvar_hq.parquet",
        "output_dir": "D:/variant_data/embeddings/test_clinvar"
    }
]

# 2. Không gian ma trận các mô hình mục tiêu cần trích xuất đặc trưng
# Bắt buộc khai báo chuẩn xác model_id trên HuggingFace và phân loại kiến trúc học (mlm / causal)
MODELS_SPACE = [
    {
        "name": "nt_v1_500m",
        "model_id": "InstaDeepAI/nucleotide-transformer-v1-500m-humanref",
        "model_type": "mlm",
        "seq_type": "dna",
        "batch_size": 64
    },
    {
        "name": "nt_v3_650m",
        "model_id": "InstaDeepAI/nucleotide-transformer-v3-650m",
        "model_type": "mlm",
        "seq_type": "dna",
        "batch_size": 32
    },
    {
        "name": "evo2_1b",
        "model_id": "arc-institute/evo2-1b", # Hoặc đường dẫn checkpoint local
        "model_type": "causal",
        "seq_type": "dna",
        "batch_size": 16
    },
    {
        "name": "esm1b_650m",
        "model_id": "facebook/esm1b_t33_650M_UR50S",
        "model_type": "mlm",
        "seq_type": "protein",
        "batch_size": 32
    },
    {
        "name": "esm2_650m",
        "model_id": "facebook/esm2_t33_650M_UR50D",
        "model_type": "mlm",
        "seq_type": "protein",
        "batch_size": 32
    },
    {
        "name": "esmc_600m",
        "model_id": "evolutionaryscale/esmc-open-v0.1", 
        "model_type": "mlm",
        "seq_type": "protein",
        "batch_size": 32
    }
]

# Set up cho Module 3
BASE_DIR = r"D:/variant_data"
EMB_DIR = f"{BASE_DIR}/embeddings"       # Nơi chứa các file .pt từ Module 2
INDEX_DIR = f"{BASE_DIR}/faiss_indexes"  # Nơi lưu file .index tĩnh
GEOM_DIR = f"{BASE_DIR}/geometry"        # Nơi lưu file kết quả .parquet

# Định nghĩa các tập dữ liệu và không gian mô hình
SPLITS = ["train", "val", "test_clinvar_hq"] # Lưu ý: sửa tên thư mục test cho khớp với Module 2
MODELS = ["nt_v1_500m", "nt_v3_650m", "evo2_1b", "esm1b_650m", "esm2_650m", "esmc_600m"]
POOLINGS = ["cls", "center", "mean"]

# Khởi tạo thư mục đích
os.makedirs(INDEX_DIR, exist_ok=True)
for split in SPLITS:
    os.makedirs(f"{GEOM_DIR}/{split}", exist_ok=True)

# Set up cho Module 4
BASE_DIR = r"D:/variant_data"
INPUT_DIR = BASE_DIR                           
OUTPUT_DIR = f"{BASE_DIR}/processed_parquet"   
ARTIFACTS_DIR = f"{BASE_DIR}/normalization_artifacts" 
GEOM_DIR = f"{BASE_DIR}/geometry"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

train_bio_path = f"{INPUT_DIR}/train_split.parquet"
eval_bio_paths = [
    f"{INPUT_DIR}/val_split.parquet",
    f"{INPUT_DIR}/test_clinvar_hq.parquet"
]

# MODULE 2: FOUNDATION MODELS EMBEDDING EXTRACTION

In [ ]:
# Vòng lặp tối thượng: Duyệt qua từng mô hình -> Tải vào VRAM -> Duyệt qua tất cả các file dữ liệu
for model_cfg in MODELS_SPACE:
    print("=" * 80)
    print(f"[MÔ HÌNH HIỆN TẠI]: KHỞI CHẠY TIẾN TRÌNH TRÍCH XUẤT CHO MÔ HÌNH: {model_cfg['name'].upper()}")
    print("=" * 80)
    
    # 1. Khởi tạo Trình quản lý mô hình và đẩy thẳng lên GPU với độ chính xác Mixed Precision (FP16)
    try:
        model_manager = BioModelManager(
            model_id=model_cfg["model_id"],
            model_type=model_cfg["model_type"],
            device="cuda"
        )
        extractor = FeatureExtractor(model_manager=model_manager)
        
    except Exception as e:
        print(f"[LỖI CHIẾN LƯỢC] Không thể nạp mô hình {model_cfg['name']}. Lỗi chi tiết: {e}")
        print("Tự động bỏ qua để chuyển sang mô hình tiếp theo trong danh sách space...")
        continue

    # 2. Duyệt qua từng file dữ liệu đầu vào ứng với mô hình hiện tại
    for data_cfg in INPUT_FILES_CONFIG:
        input_path = data_cfg["input_path"]
        output_dir = data_cfg["output_dir"]
        
        # Đảm bảo thư mục đích an toàn, tạo tự động nếu chưa tồn tại
        os.makedirs(output_dir, exist_ok=True)
        
        # Thiết lập tên file đầu ra theo quy chuẩn phân biệt cấu hình
        output_prefix = os.path.join(output_dir, f"{model_cfg['name']}")
        
        print(f"\n[+] Đang xử lý tập dữ liệu: {os.path.basename(input_path)}")
        print(f"    -> Dữ liệu nguồn: {input_path}")
        print(f"    -> Tiền tố lưu trữ: {output_prefix}_[strategy].pt")
        
        # Kích hoạt Core Pipeline trích xuất (Chạy Double Forward Pass & Pooling & LLR)
        extractor.run_extraction(
            parquet_path=input_path,
            seq_type=model_cfg["seq_type"],
            batch_size=model_cfg["batch_size"],
            output_prefix=output_prefix
        )
        
        # Giải phóng dung lượng đệm GPU sau khi xử lý xong một file dữ liệu để đảm bảo an toàn bộ nhớ
        torch.cuda.empty_cache()
        gc.collect()
        
    # 3. Hủy hoàn toàn thực thể mô hình hiện tại khỏi VRAM trước khi nạp mô hình tiếp theo
    print(f"\n[+] Đang dọn dẹp và giải phóng VRAM cho cấu hình mô hình: {model_cfg['name']}")
    del model_manager
    del extractor
    torch.cuda.empty_cache()
    gc.collect()

print("\n" + "#" * 50)
print("[THÀNH CÔNG RỰC RỠ] ĐÃ HOÀN TẤT TRÍCH XUẤT ĐA PHƯƠNG THỨC CHO TẤT CẢ CÁC MÔ HÌNH NỀN TẢNG!")
print("#" * 50)

# MODULE 3: LATENT GEOMETRIC EXTRACTION

In [ ]:
# Khởi tạo bộ tính toán Hình học
# Sử dụng k_neighbors=32 và epsilon=1e-8 theo chuẩn thiết kế
geom_calculator = LatentGeometryCalculator(k_neighbors=32, epsilon=1e-8)

print("=" * 80)
print("[*] KHỞI CHẠY QUÁ TRÌNH TRÍCH XUẤT HÌNH HỌC TIỀM ẨN (LVD & LID)")
print("=" * 80)

# ---------------------------------------------------------
# VÒNG LẶP XỬ LÝ (BATCH RUNNER)
# ---------------------------------------------------------
for model_name in MODELS:
    for pooling in POOLINGS:
        config_id = f"{model_name}_{pooling}"
        print(f"\n[>>>] ĐANG XỬ LÝ TỔ HỢP: {config_id.upper()} [<<<]")
        
        # --- BƯỚC A: HUẤN LUYỆN GLOBAL FAISS INDEX TỪ TẬP TRAIN ---
        train_pt_path = f"{EMB_DIR}/train/{config_id}.pt"
        index_path = f"{INDEX_DIR}/{config_id}.index"
        
        if not os.path.exists(train_pt_path):
            print(f"[!] Bỏ qua tổ hợp {config_id} vì không tìm thấy file Train Embeddings.")
            continue
            
        print("  -> Đang nạp tập Train để build FAISS Index...")
        train_data = torch.load(train_pt_path, weights_only=False)
        e_ref_train = train_data["E_ref"]
        e_alt_train = train_data["E_alt"]
        
        # Tính toán vector dịch chuyển Delta (E_alt - E_ref) và ép về numpy float32
        delta_train = (e_alt_train - e_ref_train).cpu().numpy().astype(np.float32)
        
        # Build và Save Index
        geom_calculator.build_and_save_global_index(delta_train=delta_train, index_path=index_path)
        
        # Nạp lại Index thẳng lên GPU để chuẩn bị Inference
        geom_calculator.load_global_index(index_path=index_path)
        
        # Giải phóng RAM lập tức cho biến train
        del train_data, e_ref_train, e_alt_train, delta_train
        torch.cuda.empty_cache()
        gc.collect()

        # --- BƯỚC B: TRÍCH XUẤT HÌNH HỌC CHO MỌI TẬP (TRAIN, VAL, TEST) ---
        for split in SPLITS:
            pt_path = f"{EMB_DIR}/{split}/{config_id}.pt"
            out_parquet_path = f"{GEOM_DIR}/{split}/{config_id}_geom.parquet"
            
            if not os.path.exists(pt_path):
                print(f"    [!] Bỏ qua tập {split} vì không thấy file {pt_path}")
                continue
                
            print(f"  -> Đang trích xuất đặc trưng cho tập: {split.upper()}...")
            
            # 1. Nạp dữ liệu
            split_data = torch.load(pt_path, weights_only=False)
            metadata = split_data["metadata"]
            llr = split_data["llr"].cpu().numpy().flatten()
            e_ref = split_data["E_ref"].cuda()
            e_alt = split_data["E_alt"].cuda()
            
            # 2. Đưa qua Lăng kính Hình học
            # Chạy on-the-fly qua class LatentGeometryCalculator
            geom_features = geom_calculator.extract_geometry_features(e_ref, e_alt)
            
            # 3. Chuyển đổi Tensor thành Pandas DataFrame
            df_geom = pd.DataFrame({
                "Variant_ID": metadata,
                "LLR": llr.astype(np.float32),
                "LVD_L2": geom_features["LVD_L2"].cpu().numpy().flatten().astype(np.float32),
                "LVD_Cosine": geom_features["LVD_Cosine"].cpu().numpy().flatten().astype(np.float32),
                "LID": geom_features["LID"].cpu().numpy().flatten().astype(np.float32)
            })
            
            # 4. Lưu thành định dạng Parquet siêu nhẹ
            df_geom.to_parquet(out_parquet_path, index=False)
            print(f"      [+] Đã lưu: {out_parquet_path} (Kích thước: {df_geom.shape})")
            
            # 5. Dọn dẹp VRAM theo từng Split
            del split_data, e_ref, e_alt, geom_features, df_geom
            torch.cuda.empty_cache()
            gc.collect()

        # --- BƯỚC C: GIẢI PHÓNG FAISS INDEX KHỎI GPU ---
        geom_calculator.index = None
        torch.cuda.empty_cache()
        gc.collect()

print("\n" + "=" * 80)
print("[THÀNH CÔNG] ĐÃ HOÀN TẤT TOÀN BỘ QUÁ TRÌNH TRÍCH XUẤT HÌNH HỌC VÀ LƯU THÀNH PARQUET!")
print("=" * 80)

# MODULE 4: BIOLOGICAL CONTEXT NORMALIZATION

In [ ]:
# Khởi tạo class
normalizer = AdvancedFeatureNormalizer(epsilon=1e-8)

print("=" * 80)
print("[*] KHỞI CHẠY QUÁ TRÌNH CHUẨN HÓA DỮ LIỆU BẢNG (BIO & GEOMETRY)")
print("=" * 80)

# ==============================================================================
# PHẦN 1: CHUẨN HÓA DỮ LIỆU SINH HỌC (BIO CONTEXT)
# ==============================================================================
# 1A. Fit & Transform Train
print("  -> Học quy tắc chuẩn hóa từ tập Train (Bio)...")
df_train_bio = pd.read_parquet(train_bio_path)
df_train_bio_norm = normalizer.fit_transform_bio(df_train_bio, ARTIFACTS_DIR)
df_train_bio_norm.to_parquet(f"{OUTPUT_DIR}/train_normalized.parquet", index=False)

# 1B. Transform Eval
for eval_path in eval_bio_paths:
    print(f"  -> Áp dụng chuẩn hóa cho tập Eval (Bio): {os.path.basename(eval_path)}...")
    df_eval_bio = pd.read_parquet(eval_path)
    df_eval_bio_norm = normalizer.transform_bio(df_eval_bio, ARTIFACTS_DIR)
    
    file_name = os.path.basename(eval_path).replace(".parquet", "_normalized.parquet")
    df_eval_bio_norm.to_parquet(f"{OUTPUT_DIR}/{file_name}", index=False)


# ==============================================================================
# PHẦN 2: CHUẨN HÓA DỮ LIỆU HÌNH HỌC (GEOMETRIC CONTEXT) - LẶP 18 CẤU HÌNH
# ==============================================================================
print("\n[>>>] PHẦN 2: CHUẨN HÓA ĐẶC TRƯNG HÌNH HỌC [<<<]")
MODELS = ["nt_v1_500m", "nt_v3_650m", "evo2_1b", "esm1b_650m", "esm2_650m", "esmc_600m"]
POOLINGS = ["cls", "center", "mean"]
SPLITS = ["train", "val", "test_clinvar_hq"]

for model_name in MODELS:
    for pooling in POOLINGS:
        config_name = f"{model_name}_{pooling}"
        print(f"\n  [Tổ hợp: {config_name.upper()}]")
        
        # 2A. Fit & Transform Train cho cấu hình hiện tại
        train_geom_path = f"{GEOM_DIR}/train/{config_name}_geom.parquet"
        if not os.path.exists(train_geom_path):
            continue
            
        print("    -> Fit & Transform Train...")
        df_train_geom = pd.read_parquet(train_geom_path)
        df_train_geom_norm = normalizer.fit_transform_geom(df_train_geom, ARTIFACTS_DIR, config_name)
        
        out_train_geom = f"{GEOM_DIR}/train/{config_name}_geom_norm.parquet"
        df_train_geom_norm.to_parquet(out_train_geom, index=False)
        
        # [BẢN VÁ] Dọn rác ngay sau khi xong tập Train
        del df_train_geom, df_train_geom_norm
        gc.collect()
        
        # 2B. Transform Eval cho cấu hình hiện tại
        for split in SPLITS[1:]: # Lấy Val và Test
            eval_geom_path = f"{GEOM_DIR}/{split}/{config_name}_geom.parquet"
            if not os.path.exists(eval_geom_path):
                continue
                
            print(f"    -> Transform {split.upper()}...")
            df_eval_geom = pd.read_parquet(eval_geom_path)
            df_eval_geom_norm = normalizer.transform_geom(df_eval_geom, ARTIFACTS_DIR, config_name)
            
            out_eval_geom = f"{GEOM_DIR}/{split}/{config_name}_geom_norm.parquet"
            df_eval_geom_norm.to_parquet(out_eval_geom, index=False)
            
            # [BẢN VÁ] Dọn rác ngay sau khi xong từng tập Eval
            del df_eval_geom, df_eval_geom_norm
            gc.collect()

print("\n" + "=" * 80)
print("[THÀNH CÔNG] DỮ LIỆU ĐÃ ĐƯỢC CHUẨN HÓA TOÀN DIỆN VÀ SẴN SÀNG CHO FUSION (MODULE 5)!")
print("=" * 80)